# 16 主成分分析 PCA

PCA 是经典降维方法。它寻找数据中方差最大的方向，把高维数据投影到这些方向上。


## 0. 学习目标和阅读地图

PCA 的重点是“用少数方向保留尽量多的方差”。你需要掌握：

1. 主成分为什么是方差最大的方向。
2. explained variance ratio 怎么读。
3. PCA 和监督学习的区别。
4. loading 如何解释主成分。


## 1. 数学逻辑

先中心化数据：

$$X_c = X - mean(X)$$

协方差矩阵：

$$C = \frac{1}{n-1}X_c^TX_c$$

PCA 寻找协方差矩阵最大的特征值对应的特征向量：

$$Cv = \lambda v$$

最大的 `lambda` 表示该方向解释的方差最多。


## 1.1 推导拆开看：最大方差方向

如果把数据投影到单位向量 `v` 上，投影后的方差是：

$$Var(Xv)=v^TCv$$

PCA 的第一主成分就是求：

$$\max_{||v||=1}v^TCv$$

这个优化问题的解是协方差矩阵最大特征值对应的特征向量。

第二主成分是在和第一主成分正交的限制下，继续找方差最大的方向。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager


def setup_chinese_font():
    candidates = [
        'PingFang SC',
        'Heiti SC',
        'Songti SC',
        'Arial Unicode MS',
        'Noto Sans CJK SC',
        'Noto Sans SC',
        'SimHei',
        'Microsoft YaHei',
        'WenQuanYi Micro Hei',
    ]
    available = {font.name for font in font_manager.fontManager.ttflist}
    for font in candidates:
        if font in available:
            existing = [name for name in plt.rcParams['font.sans-serif'] if name != font]
            plt.rcParams['font.family'] = 'sans-serif'
            plt.rcParams['font.sans-serif'] = [font] + existing
            break
    else:
        print('Warning: no Chinese font found. Install Noto Sans CJK SC or SimHei if Chinese text is missing in plots.')
    plt.rcParams['axes.unicode_minus'] = False


setup_chinese_font()

from sklearn.decomposition import PCA
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
data = load_iris()
X = data.data
y = data.target
X_s = StandardScaler().fit_transform(X)


## 1.2 标准化和 loading

Iris 数据有 4 个特征，量纲和尺度不同。PCA 之前标准化，是为了避免某个数值范围大的特征主导方差。

`components_` 可以看作主成分的 loading：它告诉你每个原始特征在主成分里的权重。


In [ ]:
# 从零实现 PCA：中心化 -> 协方差 -> 特征分解 -> 投影
Xc = X_s - X_s.mean(axis=0)
cov = Xc.T @ Xc / (len(Xc) - 1)
eigvals, eigvecs = np.linalg.eigh(cov)
order = np.argsort(eigvals)[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]
Z = Xc @ eigvecs[:, :2]

print('解释方差比例:', np.round(eigvals[:2] / eigvals.sum(), 3))
plt.scatter(Z[:,0], Z[:,1], c=y, cmap='viridis', edgecolor='k')
plt.title('从零 PCA 投影到二维')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()


## 1.3 从零实现代码怎么读

从零 PCA 的顺序：

1. `Xc = X_s - mean`：中心化。
2. `cov = Xc.T @ Xc / (n-1)`：计算协方差。
3. `np.linalg.eigh(cov)`：求特征值和特征向量。
4. 按特征值从大到小排序。
5. `Z = Xc @ eigvecs[:, :2]`：投影到前两个主成分。


In [ ]:
model = PCA(n_components=2)
Z2 = model.fit_transform(X_s)
print('sklearn explained_variance_ratio:', np.round(model.explained_variance_ratio_, 3))

plt.scatter(Z2[:,0], Z2[:,1], c=y, cmap='viridis', edgecolor='k')
plt.title('sklearn PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()


In [ ]:
# 诊断：累计解释方差和主成分 loading
pca_full = PCA().fit(X_s)
cum = np.cumsum(pca_full.explained_variance_ratio_)
plt.plot(range(1, len(cum) + 1), cum, marker='o')
plt.ylim(0, 1.05)
plt.title('累计解释方差')
plt.xlabel('主成分数量')
plt.ylabel('cumulative explained variance')
plt.show()

print('前两个主成分 loading:')
for i, comp in enumerate(model.components_, 1):
    print(f'PC{i}:', dict(zip(data.feature_names, np.round(comp, 3))))


## 2.1 如何诊断 PCA

PCA 常看累计解释方差。如果前两个主成分只解释很少方差，那么二维图可能丢失大量信息。

解释主成分时不要只看散点图，还要看 loading：哪些原始特征贡献了这个方向。


## 2. 常见误区

- PCA 是无监督方法，不会主动寻找最利于分类的方向。
- PCA 对特征尺度敏感，通常需要标准化。
- 主成分是原始特征的线性组合，解释时要看 loading。

## 3. 小实验

- 改 `n_components`，观察累计解释方差。
- 不做标准化再跑 PCA。
- 在二维投影上训练一个分类器。


## 5. 复习清单

- PCA 是无监督降维，不使用标签。
- 主成分是互相正交的方向。
- 标准化通常很重要。
- explained variance 高不代表对分类任务最有用。
